# 从零实现 Transformer 核心机制

让我们像 2017 年的研究者一样，带着问题、公式和草稿纸，从零开始发明 Transformer 的核心机制。我们将一步步追问：为什么要自注意力？为什么是点积？为什么要除以 √d_k ？每一处都会先给出思路，再写出数学表达式，最后用最基础的张量操作实现并观察输出，绝不使用 `nn.MultiheadAttention` 这样的成品工具。

---

## 1. 从 RNN 的枷锁到"让每个词直接看到所有词"

**Idea**
循环神经网络（RNN）必须按顺序读句子：第 t 步的隐藏状态必须等第 t-1 步算完。这带来了两个痛苦：
- 无法并行训练。
- 长距离依赖被多次乘法削弱。

我们希望每个位置的表示能**直接与序列中所有位置交互**，并把那些最相关位置的信息"汇总"过来。一种自然的想法：用内积来衡量两个词向量的相关性，相关性越高，分配越多的注意力。

**Mathematical expression**
假设我们有序列的表征矩阵 $X \in \mathbb{R}^{n \times d}$，其中 n 为序列长度，d 为每个词的向量维度。我们定义一个相关性矩阵：
$$
\text{scores} = X X^\top \quad (\text{大小 } n \times n)
$$
矩阵的第 i 行第 j 列就是词 i 与词 j 的内积相似度。随后用 softmax 得到归一化的注意力权重，并用它去加权求和 X 自身：
$$
\text{Attention}(X) = \text{softmax}(X X^\top) X
$$
但目前这个版本有一个致命问题。

**Code & Output**
我们先用 PyTorch 试试看：

In [ ]:
import torch
import torch.nn.functional as F

# 假设序列长度 n=4，每个词向量维度 d=64
torch.manual_seed(42)
n, d = 4, 64
X = torch.randn(n, d)  # 随机初始化一个序列表征

# 计算未缩放的点积分数
scores = X @ X.T        # @ 表示矩阵乘法，形状 [4,4]
print("未缩放分数矩阵:\n", scores)
# 观察：某些分数绝对值已经到 20~30，softmax 后会极其尖锐

接着计算 softmax 权重，并查看某一行的分布：

In [ ]:
weights = F.softmax(scores, dim=-1)  # 沿每一行做 softmax
print("Softmax 权重（第0行）:", weights[0])
# 很可能看到类似 [0.9999, 0, 0, 0]，梯度几乎为零

**为什么这会失败？**
当 d 较大时，随机向量的点积方差约为 d。如果 q 和 k 的分量独立同分布，均值为 0，方差为 1，则 $\text{Var}(q \cdot k) = d$。点积值幅度随 d 增大而增大，经过 softmax 后几乎变成 one-hot，导致梯度消失。

**改进**
我们让点积除以 $\sqrt{d}$ 来将方差重新控制为 1：
$$
\text{Attention}(X) = \text{softmax}\left(\frac{X X^\top}{\sqrt{d}}\right) X
$$
这就是"缩放点积"的由来。此时 softmax 的输入不再过大，权重变得平缓。

In [ ]:
d_k = d  # 此时键向量的维度
scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float))
weights_scaled = F.softmax(scaled_scores, dim=-1)
print("缩放后 Softmax 权重（第0行）:", weights_scaled[0])
# 现在权重分布更合理，例如 [0.2, 0.3, 0.1, 0.4]

这一步写完，我们手里就有了最简陋的"缩放点积自注意力"——每一行代码的目的：  
- `scores = X @ X.T`：获得所有词对之间的原始相似度。  
- `torch.sqrt(torch.tensor(d_k))`：计算缩放因子 $\sqrt{d_k}$。  
- `scaled_scores`：让点积的方差重新回到 1，防止 softmax 饱和。  
- `F.softmax(scaled_scores, dim=-1)`：将每行的相似度转化为概率分布（注意力权重）。  
- `weights_scaled @ X`（稍后会做）：用这些概率去取 X 中其他词的加权平均。

---

## 2. 引入可学习的 Q、K、V：让网络学会"查什么、键是什么、取什么"

**Idea**
上一步里，我们直接用 X 本身充当了"用来查询的词""被查询的键"以及"被聚合的值"。但模型需要灵活地学习不同的表示空间：  
- **查询 Q**：代表"我在找什么"。  
- **键 K**：代表"我含有什么信息"。  
- **值 V**：代表"如果你选中了我，我将提供什么内容"。  

为此，我们引入三个可训练的权重矩阵 $W_Q, W_K, W_V$，将原始输入 X 投影到不同角色。

**Mathematical expression**
给定 $X \in \mathbb{R}^{n \times d_{\text{model}}}$：
$$
Q = X W_Q \quad (W_Q \in \mathbb{R}^{d_{\text{model}} \times d_k})
$$
$$
K = X W_K \quad (W_K \in \mathbb{R}^{d_{\text{model}} \times d_k})
$$
$$
V = X W_V \quad (W_V \in \mathbb{R}^{d_{\text{model}} \times d_v})
$$
注意力计算变为：
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$
注意 Q 和 K 的维度必须相同（都为 $d_k$），才能做内积；V 的维度 $d_v$ 可以是任意值，我们常取 $d_v = d_k$。

**Code & Output — 手动创建可训练权重并实现**
我们不用 `nn.Linear` 的一步到位，而是显式定义 $W_Q, W_K, W_V$ 矩阵，并追踪梯度。

In [ ]:
torch.manual_seed(42)
n, d_model, d_k, d_v = 4, 64, 64, 64

X = torch.randn(n, d_model)          # [4, 64]

# 手动初始化权重矩阵，并设为需要梯度
W_Q = torch.randn(d_model, d_k, requires_grad=True)   # [64, 64]
W_K = torch.randn(d_model, d_k, requires_grad=True)
W_V = torch.randn(d_model, d_v, requires_grad=True)

# 计算 Q, K, V —— 每行都是 X 的线性变换
Q = X @ W_Q   # [4, 64]
K = X @ W_K   # [4, 64]
V = X @ W_V   # [4, 64]

print("Q 形状:", Q.shape)  # torch.Size([4, 64])

接下来计算缩放点积注意力，仍然要除以 $\sqrt{d_k}$：

In [ ]:
# 计算注意力分数矩阵 (Q 与 K 转置相乘)
scores = Q @ K.T                     # [4, 4] — 每个查询与所有键的相似度
# 缩放
d_k_tensor = torch.tensor(d_k, dtype=torch.float)
scaled_scores = scores / torch.sqrt(d_k_tensor)  # 仍为 [4, 4]
# softmax 得到注意力权重
attn_weights = F.softmax(scaled_scores, dim=-1)   # 每行概率和为 1
print("注意力权重:\n", attn_weights)
# 加权聚合值向量
output = attn_weights @ V            # [4, 64]
print("注意力输出形状:", output.shape)  # [4, 64]

**逐行解释：**  
- `W_Q, W_K, W_V` 是模型学习的参数，把输入映射到不同的空间。  
- `Q = X @ W_Q`：X 中每个词向量与 W_Q 相乘，得到对应的查询向量。  
- `K = X @ W_K`：得到键向量，用于被查询。  
- `V = X @ W_V`：得到值向量，最终的上下文信息将从 V 中抽取。  
- `scores = Q @ K.T`：每个查询与所有键计算点积，得到一个 n×n 的相关矩阵。  
- 除以 `sqrt(d_k)`：保证输入 softmax 的数值尺度稳定。  
- `F.softmax(..., dim=-1)`：沿键的方向归一化，得到每个查询对所有键的注意力分布。  
- `attn_weights @ V`：用注意力概率去取 V 中的信息，完成"软寻址"。

**此时我们完全拥有了 Transformer 中最核心的缩放点积注意力，并且它是可学习的。**

---

## 3. 多头：用多个"视角"同时关注不同位置

**Idea**
只用一个 Q/K/V 投影，模型可能会混合不同层面的信息（比如既想学语法又想学语义），导致注意力含糊。研究者的灵感是：并行做多组注意力，每组维度缩小，最后拼接。这就像 CNN 中多个卷积核提取不同特征。

**Mathematical expression**
设头数 h，每个头的维度 $d_k = d_v = d_{\text{model}} / h$。对每个头 i：
$$
\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)
$$
其中 $W_i^Q \in \mathbb{R}^{d_{\text{model}} \times d_k}$ 等。然后将所有头拼接：
$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W_O
$$
$W_O \in \mathbb{R}^{h d_v \times d_{\text{model}}}$ 将拼接的结果重新投影回原始维度。

**实现技巧**：把多个头的计算合并成一个批量矩阵乘法，而不是用 for 循环。我们将 Q、K、V 线性变换后重塑成 `[batch, n, h, d_k]`，然后转置为 `[batch, h, n, d_k]`，并行计算所有头的注意力。

**Code & Output**

In [ ]:
torch.manual_seed(42)
n, d_model, h = 4, 64, 8        # 8 个头
d_k = d_model // h               # 每个头维度 8

X = torch.randn(n, d_model)

# 定义整体的 Q, K, V 投影矩阵（合并所有头的权重）
W_Q_all = torch.randn(d_model, h * d_k, requires_grad=True)   # [64, 64]
W_K_all = torch.randn(d_model, h * d_k, requires_grad=True)
W_V_all = torch.randn(d_model, h * d_k, requires_grad=True)

# 1. 投影并重塑
Q_all = X @ W_Q_all                      # [4, 64] -> [n, h*d_k]
# reshape 成 [n, h, d_k]，然后为了并行计算，交换维度为 [h, n, d_k]
Q = Q_all.view(n, h, d_k).transpose(0, 1)   # [8, 4, 8]
K = (X @ W_K_all).view(n, h, d_k).transpose(0, 1)  # 同上
V = (X @ W_V_all).view(n, h, d_k).transpose(0, 1)

# 2. 计算缩放点积注意力，在一个批次内对所有头进行
# Q: [h, n, d_k], K: [h, n, d_k] -> K 转置最后两维 [h, d_k, n]
scores = torch.matmul(Q, K.transpose(-2, -1))  # [h, n, n]
scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float))
attn_weights = F.softmax(scaled_scores, dim=-1) # [h, n, n]

# 3. 用注意力权重聚合 V
head_outputs = torch.matmul(attn_weights, V)   # [h, n, d_k]

# 4. 拼接头并做最终投影
# 先将 [h, n, d_k] 转回 [n, h, d_k] 然后变成 [n, h*d_k]
concat = head_outputs.transpose(0, 1).contiguous().view(n, h * d_k)  # [4, 64]

W_O = torch.randn(h * d_k, d_model, requires_grad=True)
multihead_output = concat @ W_O               # [4, 64]
print("多头注意力输出形状:", multihead_output.shape)  # torch.Size([4, 64])

**逐行要点：**  
- `W_Q_all` 的形状是 `[d_model, h*d_k]`，它其实等价于 h 个头各自的 $W_i^Q$ 横向拼接。  
- `.view(n, h, d_k).transpose(0,1)`：把"头"维度提到最前，这样后面 `matmul` 就能同时对 8 个头广播。  
- `scores = torch.matmul(Q, K.transpose(-2, -1))`：为每个头独立计算 n×n 注意力分数。  
- `softmax` 在最后一维进行，得到每个头内部的归一化权重。  
- `head_outputs.transpose(0,1).contiguous().view(...)` 把头拼回特征维度。  
- `W_O` 再投影，使输出的维度和输入 X 一致，方便堆叠下一个块。

---

## 4. 位置编码：告诉模型"第一个词"是什么意思

**Idea**
上面的注意力机制对词的顺序完全不敏感——无论输入顺序如何打乱，注意力加权求和的结果都一样。我们必须给词向量注入位置信息。  
研究者选择了正弦余弦位置编码，它不同位置的向量可以通过线性关系互相表达，有助于模型捕捉相对位置。

**Mathematical expression**
对位置 pos 和维度 i（从 0 到 d-1）：
$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)
$$
然后将这个矩阵直接加到输入 X 上。

**Code & Output**

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)            # 预分配
    position = torch.arange(0, max_len).unsqueeze(1).float()  # [max_len, 1]
    # 计算分母 10000^(2i/d)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                         (-torch.log(torch.tensor(10000.0)) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)  # 偶数位正弦
    pe[:, 1::2] = torch.cos(position * div_term)  # 奇数位余弦
    return pe

max_len, d_model = 4, 64
pe = positional_encoding(max_len, d_model)        # [4,64]
print("位置编码矩阵 (部分):\n", pe[:, :8])  # 展示前8维
X_with_pos = X + pe           # 将位置编码逐元素加到词向量上
print("加入位置编码后 X 的形状:", X_with_pos.shape)  # [4, 64]

**每行解释**  
- `position` 生成列向量 [0,1,2,3]^T。  
- `div_term` 是 $1/10000^{2i/d}$ 的对数形式，避免数值计算不稳定。  
- 偶数索引用 sin，奇数索引用 cos，使每一维对应不同频率。  
- 最后直接加到 X，让模型知道每个词的绝对（及相对）位置。

---

## 5. 一个完整的 Transformer 块：注意力 + 前馈 + 残差 & 层归一化

**Idea**
将前面所有的部件组合成一个编码器块。研究者发现必须加入：
- 残差连接：让信息绕道传播，缓解梯度消失。
- 层归一化：稳定训练。
- 逐位置的前馈网络（FFN）：给每个位置的表示增加非线性变换，通常由两个线性层和激活函数构成。

**Mathematical expression（一个块）**
输入 $x$：
$$
\text{attn\_out} = \text{LayerNorm}(x + \text{MultiHead}(x))
$$
$$
\text{ffn\_out} = \text{LayerNorm}(\text{attn\_out} + \text{FFN}(\text{attn\_out}))
$$
FFN 为：
$$
\text{FFN}(z) = \text{ReLU}(z W_1 + b_1) W_2 + b_2
$$

**Code & Output**  
此处我们将前面写好的多头注意力封装为函数（不使用 `nn.MultiheadAttention`），并构建一个块。

In [ ]:
# 手动实现多头注意力（加入位置编码后），用前面的方法
def multihead_attention(X, W_Q_all, W_K_all, W_V_all, W_O, h, d_k):
    n = X.size(0)
    Q = (X @ W_Q_all).view(n, h, d_k).transpose(0, 1)
    K = (X @ W_K_all).view(n, h, d_k).transpose(0, 1)
    V = (X @ W_V_all).view(n, h, d_k).transpose(0, 1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float))
    attn_weights = F.softmax(scores, dim=-1)
    head_outs = torch.matmul(attn_weights, V)
    concat = head_outs.transpose(0,1).contiguous().view(n, h * d_k)
    return concat @ W_O

# 参数初始化
torch.manual_seed(42)
n, d_model, h = 4, 64, 8
d_k = d_model // h
X = torch.randn(n, d_model)
pe = positional_encoding(n, d_model)
X = X + pe

# 创建所有权重（可学习）
W_Q_all = torch.randn(d_model, h*d_k, requires_grad=True)
W_K_all = torch.randn(d_model, h*d_k, requires_grad=True)
W_V_all = torch.randn(d_model, h*d_k, requires_grad=True)
W_O     = torch.randn(h*d_k, d_model, requires_grad=True)

# --- 注意力子层 + 残差 & LayerNorm ---
attn_out = multihead_attention(X, W_Q_all, W_K_all, W_V_all, W_O, h, d_k)
# 残差连接并归一化（手动使用 nn.LayerNorm，这是基础的归一化工具）
layernorm1 = torch.nn.LayerNorm(d_model)
x1 = layernorm1(X + attn_out)   # [4,64]

# --- 前馈网络子层 + 残差 & LayerNorm ---
W1 = torch.randn(d_model, 4*d_model, requires_grad=True)   # FFN 通常中间维度扩大
b1 = torch.randn(4*d_model, requires_grad=True)
W2 = torch.randn(4*d_model, d_model, requires_grad=True)
b2 = torch.randn(d_model, requires_grad=True)

ffn_out = torch.relu(x1 @ W1 + b1) @ W2 + b2   # 逐位置前馈，依然是 [4,64]
layernorm2 = torch.nn.LayerNorm(d_model)
output = layernorm2(x1 + ffn_out)

print("Transformer 块最终输出形状:", output.shape)  # torch.Size([4, 64])
print("输出示例:\n", output)

**说明：**  
- `X + attn_out` 是残差连接，把原始输入直接加到注意力结果上。  
- `LayerNorm` 对每个样本的 hidden 维度做归一化，稳定训练。  
- FFN 里的 `@ W1 + b1` 把每个位置独立地映射到更高维度，经过 ReLU 再压回原维度。  
- 以上所有操作均未使用 `nn.MultiheadAttention`，而是用明确的矩阵乘法重现。

---

## 回首：我们是如何一步步走到这里的？

我们经历的路径正对应了当年研究者的思维轨迹：  
1. **想要并行化并捕捉长距离依赖** → 直接算全序列内积相似度。  
2. **内积数值太大导致 softmax 梯度消失** → 除以 √d_k 缩放，使方差归一化。  
3. **希望网络学会"关注什么"** → 引入可学习的 Q, K, V 投影。  
4. **单一注意力混杂多种信息** → 切成多个头，从不同子空间聚合。  
5. **缺失序列顺序** → 加上正弦/余弦位置编码。  
6. **最终组件化** → 用残差连接、层归一化和前馈网络搭成可堆叠的块。

每一步都是先有 idea，然后落实到数学表达式，最后用最基础的张量操作（点积、softmax、reshape、转置）变成代码，并亲眼看到张量形状和数值是否符合预期。这正是当年发明 Transformer 的过程——没有现成的 `MultiheadAttention`，只有对注意力本质的追问与干净的矩阵计算。